In [1]:
import numpy as np
import tvm
from tvm.script import tir as T


[08:54:01] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m1` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[08:54:01] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m1` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[08:54:01] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m2` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[08:54:01] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m2` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[08:54:01] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: Error: Using LLVM 19.1.7 with `-mcpu=apple-m1` is not valid in `-mtriple=arm64-apple-macos`, using default `-mcpu=generic`
[08:54:01] /workspace/tvm/src/target/llvm/llvm_instance.cc:219: E

## Element-wise Add


In [2]:
# init data
a = np.arange(16).reshape(4, 4)
b = np.arange(16, 0, -1).reshape(4, 4)

In [3]:
c_np = a + b
c_np

array([[16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16]])

In [4]:
# low-level numpy version
def lnumpy_add(a: np.ndarray, b: np.ndarray, c: np.ndarray):
    for i in range(4):
        for j in range(4):
            c[i, j] = a[i, j] + b[i, j]


c_lnumpy = np.empty((4, 4), dtype=np.int64)
lnumpy_add(a, b, c_lnumpy)
c_lnumpy


array([[16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16]])

In [5]:
# TensorIR version
@tvm.script.ir_module
class MyAdd:
    @T.prim_func
    def add(
        A: T.Buffer((4, 4), "int64"),
        B: T.Buffer((4, 4), "int64"),
        C: T.Buffer((4, 4), "int64"),
    ):
        T.func_attr({"global_symbol": "add"})
        for i, j in T.grid(4, 4):
            with T.sblock("C"):
                vi = T.axis.spatial(4, i)
                vj = T.axis.spatial(4, j)
                C[vi, vj] = A[vi, vj] + B[vi, vj]


rt_lib = tvm.compile(MyAdd, target="llvm")
a_tvm = tvm.runtime.tensor(a)
b_tvm = tvm.runtime.tensor(b)
c_tvm = tvm.runtime.tensor(np.empty((4, 4), dtype=np.int64))
rt_lib["add"](a_tvm, b_tvm, c_tvm)
np.testing.assert_allclose(c_tvm.numpy(), c_np, rtol=1e-5)


# Broadcast Add


In [6]:
# init data
a = np.arange(16).reshape(4, 4)
b = np.arange(4, 0, -1).reshape(4)

In [7]:
# numpy version
c_np = a + b
c_np

array([[ 4,  4,  4,  4],
       [ 8,  8,  8,  8],
       [12, 12, 12, 12],
       [16, 16, 16, 16]])

In [8]:
@tvm.script.ir_module
class MyAdd:
    @T.prim_func
    def add(
        A: T.Buffer((4, 4), "int64"),
        B: T.Buffer((4), "int64"),
        C: T.Buffer((4, 4), "int64"),
    ):
        T.func_attr({"global_symbol": "add", "tir.noalias": True})
        for i, j in T.grid(4, 4):
            with T.sblock("C"):
                vi = T.axis.spatial(4, i)
                vj = T.axis.spatial(4, j)
                C[vi, vj] = A[vi, vj] + B[vj]


rt_lib = tvm.compile(MyAdd, target="llvm")
a_tvm = tvm.runtime.tensor(a)
b_tvm = tvm.runtime.tensor(b)
c_tvm = tvm.runtime.tensor(np.empty((4, 4), dtype=np.int64))
rt_lib["add"](a_tvm, b_tvm, c_tvm)
np.testing.assert_allclose(c_tvm.numpy(), c_np, rtol=1e-5)

## 2D Convolution


In [9]:
N, CI, H, W, CO, K = 1, 1, 8, 8, 2, 3
OUT_H, OUT_W = H - K + 1, W - K + 1
data = np.arange(N * CI * H * W).reshape(N, CI, H, W)
weight = np.arange(CO * CI * K * K).reshape(CO, CI, K, K)

In [10]:
# torch version
import torch

data_torch = torch.Tensor(data)
weight_torch = torch.Tensor(weight)
conv_torch = torch.nn.functional.conv2d(data_torch, weight_torch)
conv_torch = conv_torch.numpy().astype(np.int64)
conv_torch

array([[[[ 474,  510,  546,  582,  618,  654],
         [ 762,  798,  834,  870,  906,  942],
         [1050, 1086, 1122, 1158, 1194, 1230],
         [1338, 1374, 1410, 1446, 1482, 1518],
         [1626, 1662, 1698, 1734, 1770, 1806],
         [1914, 1950, 1986, 2022, 2058, 2094]],

        [[1203, 1320, 1437, 1554, 1671, 1788],
         [2139, 2256, 2373, 2490, 2607, 2724],
         [3075, 3192, 3309, 3426, 3543, 3660],
         [4011, 4128, 4245, 4362, 4479, 4596],
         [4947, 5064, 5181, 5298, 5415, 5532],
         [5883, 6000, 6117, 6234, 6351, 6468]]]])

In [11]:
conv_torch.shape

(1, 2, 6, 6)

In [12]:
data.shape, weight.shape

((1, 1, 8, 8), (2, 1, 3, 3))

In [13]:
@tvm.script.ir_module
class MyConv:
    @T.prim_func
    def conv(
        data: T.Buffer((1, 1, 8, 8), "int64"),
        weight: T.Buffer((2, 1, 3, 3), "int64"),
        conv: T.Buffer((1, 2, 6, 6), "int64"),
    ):
        T.func_attr({"global_symbol": "conv", "tirx.noalias": True})
        for n, co, h, w in T.grid(1, 2, 6, 6):
            with T.sblock("conv"):
                vn = T.axis.spatial(1, n)
                vco = T.axis.spatial(2, co)
                vh = T.axis.spatial(6, h)
                vw = T.axis.spatial(6, w)
                conv[vn, vco, vh, vw] = 0
                for ci in range(1):
                    for kh in range(3):
                        for kw in range(3):
                            conv[vn, vco, vh, vw] += (
                                data[vn, ci, vh + kh, vw + kw] * weight[vco, ci, kh, kw]
                            )

    @T.prim_func
    def conv_opt(
        data: T.Buffer((1, 1, 8, 8), "int64"),
        weight: T.Buffer((2, 1, 3, 3), "int64"),
        conv: T.Buffer((1, 2, 6, 6), "int64"),
    ):
        T.func_attr({"global_symbol": "conv_opt", "tirx.noalias": True})
        for n, co, h, w, ci, kh, kw in T.grid(1, 2, 6, 6, 1, 3, 3):
            with T.sblock("conv"):
                vn = T.axis.spatial(1, n)
                vco = T.axis.spatial(2, co)
                vh = T.axis.spatial(6, h)
                vw = T.axis.spatial(6, w)
                vci = T.axis.reduce(1, ci)
                vkh = T.axis.reduce(3, kh)
                vkw = T.axis.reduce(3, kw)
                with T.init():
                    conv[vn, vco, vh, vw] = 0
                conv[vn, vco, vh, vw] += (
                    data[vn, vci, vh + vkh, vw + vkw] * weight[vco, vci, vkh, vkw]
                )


rt_lib = tvm.compile(MyConv, target="llvm")
data_tvm = tvm.runtime.tensor(data)
weight_tvm = tvm.runtime.tensor(weight)
conv_tvm = tvm.runtime.tensor(np.empty((N, CO, OUT_H, OUT_W), dtype=np.int64))
rt_lib["conv"](data_tvm, weight_tvm, conv_tvm)
np.testing.assert_allclose(conv_tvm.numpy(), conv_torch, rtol=1e-5)

rt_lib["conv_opt"](data_tvm, weight_tvm, conv_tvm)
np.testing.assert_allclose(conv_tvm.numpy(), conv_torch, rtol=1e-5)

ex_conv = rt_lib.mod.time_evaluator("conv", tvm.cpu(0), number=100)
ex_conv_opt = rt_lib.mod.time_evaluator("conv_opt", tvm.cpu(0), number=100)
print("conv:", ex_conv(data_tvm, weight_tvm, conv_tvm).mean * 1e3, "ms")
print("conv_opt:", ex_conv_opt(data_tvm, weight_tvm, conv_tvm).mean * 1e3, "ms")

conv: 0.00038392 ms
conv_opt: 0.00025857999999999995 ms


### Parallel, Vectorize and Unroll


In [14]:
import IPython


@tvm.script.ir_module
class MyAdd:
    @T.prim_func
    def add(
        A: T.Buffer((4, 4), "int64"),
        B: T.Buffer((4, 4), "int64"),
        C: T.Buffer((4, 4), "int64"),
    ):
        T.func_attr({"global_symbol": "add"})
        for i, j in T.grid(4, 4):
            with T.sblock("C"):
                vi = T.axis.spatial(4, i)
                vj = T.axis.spatial(4, j)
                C[vi, vj] = A[vi, vj] + B[vi, vj]


sch = tvm.s_tir.Schedule(MyAdd)
block = sch.get_sblock("C", func_name="add")
i, j = sch.get_loops(block)
i0, i1 = sch.split(i, factors=[2, 2])
sch.parallel(i0)
sch.unroll(i1)
sch.vectorize(j)
IPython.display.Code(sch.mod.script(), language="python")


# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def add(A: T.Buffer((4, 4), "int64"), B: T.Buffer((4, 4), "int64"), C: T.Buffer((4, 4), "int64")):
        # with T.sblock("root"):
        for i_0 in T.parallel(2):
            for i_1 in T.unroll(2):
                for j in T.vectorized(4):
                    with T.sblock("C"):
                        vi = T.axis.spatial(4, i_0 * 2 + i_1)
                        vj = T.axis.spatial(4, j)
                        T.reads(A[vi, vj], B[vi, vj])
                        T.writes(C[vi, vj])
                        C[vi, vj] = A[vi, vj] + B[vi, vj]

### Transforming a bactch matmul program


In [15]:
def lnumpy_mm_relu_v2(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    Y = np.empty((16, 128, 128), dtype="float32")
    for n in range(16):
        for i in range(128):
            for j in range(128):
                for k in range(128):
                    if k == 0:
                        Y[n, i, j] = 0
                    Y[n, i, j] = Y[n, i, j] + A[n, i, k] * B[n, k, j]
    for n in range(16):
        for i in range(128):
            for j in range(128):
                C[n, i, j] = max(Y[n, i, j], 0)


In [20]:
a_np = np.random.rand(16, 128, 128).astype("float32")
b_np = np.random.rand(16, 128, 128).astype("float32")
c_np = np.empty((16, 128, 128), dtype="float32")
lnumpy_mm_relu_v2(a_np, b_np, c_np)

In [27]:
@tvm.script.ir_module
class MyBmmRelu:
    @T.prim_func
    def bmm_relu(
        A: T.Buffer((16, 128, 128), "float32"),
        B: T.Buffer((16, 128, 128), "float32"),
        C: T.Buffer((16, 128, 128), "float32"),
    ) -> None:
        T.func_attr({"global_symbol": "bmm_relu", "tirx.noalias": True})
        Y = T.alloc_buffer((16, 128, 128), dtype="float32")
        for n, i, j, k in T.grid(16, 128, 128, 128):
            with T.sblock("Y"):
                vn = T.axis.spatial(16, n)
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j)
                vk = T.axis.reduce(128, k)
                with T.init():
                    Y[vn, vi, vj] = 0.0
                Y[vn, vi, vj] += A[vn, vi, vk] * B[vn, vk, vj]
        for n, i, j in T.grid(16, 128, 128):
            with T.sblock("C"):
                vn = T.axis.spatial(16, n)
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j)
                C[vn, vi, vj] = T.max(Y[vn, vi, vj], 0.0)

In [26]:
before_rt_lib = tvm.compile(MyBmmRelu, target="llvm")
a_tvm = tvm.runtime.tensor(a_np)
b_tvm = tvm.runtime.tensor(b_np)
c_tvm = tvm.runtime.tensor(c_np)
before_rt_lib["bmm_relu"](a_tvm, b_tvm, c_tvm)

np.testing.assert_allclose(c_tvm.numpy(), c_np, rtol=1e-5)

In [50]:
sch = tvm.s_tir.Schedule(MyBmmRelu)

# Step 1. Get blocks
block_Y = sch.get_sblock("Y", func_name="bmm_relu")
block_C = sch.get_sblock("C", func_name="bmm_relu")

# Step 2. Get loops
n, i, j, k = sch.get_loops(block_Y)

# Step 3. Split j into (16, 8) and k into (32, 4)
j0, j1 = sch.split(j, factors=[16, 8])
k0, k1 = sch.split(k, factors=[32, 4])

# Step 4. Reorder to get n, i, j0, k0, k1, j1
sch.reorder(n, i, j0, k0, k1, j1)

# Step 5. Fuse C under j0
sch.reverse_compute_at(block_C, j0)
sch.parallel(n)

# Step 6. Decompose reduction to separate Y_init from Y_update
sch.decompose_reduction(block_Y, k0)

# Step 7. Now get fresh loop handles after decompose
block_Y_init = sch.get_sblock("Y_init", func_name="bmm_relu")
block_Y_update = sch.get_sblock("Y_update", func_name="bmm_relu")

_, _, _, ax0_init = sch.get_loops(block_Y_init)
_, _, _, ax1_0, ax1_1, ax0 = sch.get_loops(block_Y_update)
_, _, _, i2_1 = sch.get_loops(block_C)
n_loop = sch.get_loops(block_Y_init)[0]

# Step 8. Apply parallel, vectorize, unroll
sch.vectorize(ax0_init)
sch.unroll(ax1_1)
sch.vectorize(i2_1)

IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def bmm_relu(A: T.Buffer((16, 128, 128), "float32"), B: T.Buffer((16, 128, 128), "float32"), C: T.Buffer((16, 128, 128), "float32")):
        T.func_attr({"tirx.noalias": True})
        # with T.sblock("root"):
        Y = T.alloc_buffer((16, 128, 128))
        for n in T.parallel(16):
            for i, j_0 in T.grid(128, 16):
                for j_1_init in T.vectorized(8):
                    with T.sblock("Y_init"):
                        vn, vi = T.axis.remap("SS", [n, i])
                        vj = T.axis.spatial(128, j_0 * 8 + j_1_init)
                        T.reads()
                        T.writes(Y[vn, vi, vj])
                        Y[vn, vi, vj] = T.float32(0.0)
                for k_0 in range(32):
                    for k_1 in T.unroll(4):
                        for j_1 in range(8):
                            with T.sblock("Y_update"):
                                vn, vi = T.axis.remap("SS", [n, i])
                                vj = T.axis.spatial(128, j_0 * 8 + j_1)
                                vk = T.axis.reduce(128, k_0 * 4 + k_1)
                                T.reads(Y[vn, vi, vj], A[vn, vi, vk], B[vn, vk, vj])
                                T.writes(Y[vn, vi, vj])
                                Y[vn, vi, vj] = Y[vn, vi, vj] + A[vn, vi, vk] * B[vn, vk, vj]
                for ax0 in T.vectorized(8):
                    with T.sblock("C"):
                        vn, vi = T.axis.remap("SS", [n, i])
                        vj = T.axis.spatial(128, j_0 * 8 + ax0)
                        T.reads(Y[vn, vi, vj])
                        T.writes(C[vn, vi, vj])
                        C[vn, vi, vj] = T.max(Y[vn, vi, vj], T.float32(0.0))

In [51]:
@tvm.script.ir_module
class TargetModule:
    @T.prim_func
    def bmm_relu(
        A: T.Buffer((16, 128, 128), "float32"),
        B: T.Buffer((16, 128, 128), "float32"),
        C: T.Buffer((16, 128, 128), "float32"),
    ) -> None:
        T.func_attr({"global_symbol": "bmm_relu", "tirx.noalias": True})
        Y = T.alloc_buffer([16, 128, 128], dtype="float32")
        for i0 in T.parallel(16):
            for i1, i2_0 in T.grid(128, 16):
                for ax0_init in T.vectorized(8):
                    with T.sblock("Y_init"):
                        n, i = T.axis.remap("SS", [i0, i1])
                        j = T.axis.spatial(128, i2_0 * 8 + ax0_init)
                        Y[n, i, j] = T.float32(0)
                for ax1_0 in T.serial(32):
                    for ax1_1 in T.unroll(4):
                        for ax0 in T.serial(8):
                            with T.sblock("Y_update"):
                                n, i = T.axis.remap("SS", [i0, i1])
                                j = T.axis.spatial(128, i2_0 * 8 + ax0)
                                k = T.axis.reduce(128, ax1_0 * 4 + ax1_1)
                                Y[n, i, j] = Y[n, i, j] + A[n, i, k] * B[n, k, j]
                for i2_1 in T.vectorized(8):
                    with T.sblock("C"):
                        n, i = T.axis.remap("SS", [i0, i1])
                        j = T.axis.spatial(128, i2_0 * 8 + i2_1)
                        C[n, i, j] = T.max(Y[n, i, j], T.float32(0))


In [52]:
before_rt_lib = tvm.compile(MyBmmRelu, target="llvm")
after_rt_lib = tvm.compile(sch.mod, target="llvm")
a_tvm = tvm.runtime.tensor(np.random.rand(16, 128, 128).astype("float32"))
b_tvm = tvm.runtime.tensor(np.random.rand(16, 128, 128).astype("float32"))
c_tvm = tvm.runtime.tensor(np.random.rand(16, 128, 128).astype("float32"))
before_rt_lib["bmm_relu"](a_tvm, b_tvm, c_tvm)
before_timer = before_rt_lib.mod.time_evaluator("bmm_relu", tvm.cpu())
print("Before transformation:")
print(before_timer(a_tvm, b_tvm, c_tvm))

f_timer = after_rt_lib.mod.time_evaluator("bmm_relu", tvm.cpu())
print("After transformation:")
print(f_timer(a_tvm, b_tvm, c_tvm))


Before transformation:
Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
  32.3301      32.3301      32.3301      32.3301       0.0000                  
After transformation:
Execution time summary:
 mean (ms)   median (ms)    max (ms)     min (ms)     std (ms)  
   0.6204       0.6204       0.6204       0.6204       0.0000                  
